# DiD-BCF — B1_null (linearity_degree=2)

**Workstream B1 · canonical DiD (selection on unobservables)**

sharp null tau=0: size and coverage under H0

Fits DiD-BCF on the `B1_null` scenario at `linearity_degree=2` and reports
metrics for **both** the plain DiD-BCF posterior and the proposed **posterior
correction** (Algorithm 1 of the theory note), so the correction can be judged
directly. Panel: N=200, 4 pre + 4 post periods.

> **Colab:** upload just this notebook and *Run all* — the first cell installs the
> dependencies and the second clones the engine automatically.

In [1]:
# Colab: install the DiD-BCF dependencies (stochtree provides the BCF sampler).
%pip install -q stochtree scikit-learn joblib tqdm pandas numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 17.9 MB/s eta 0:00:00


In [2]:
import os, sys

# --- Locate the DiD-BCF engine ------------------------------------------------
# So you can upload just THIS notebook to Colab and Run all. Resolution order:
#   1. `did_bcf_revision` already importable;
#   2. running inside a repo checkout (the parent folder holds the package);
#   3. otherwise clone https://github.com/hugogobato/DiD-BCF and use it.
REPO_URL = "https://github.com/hugogobato/DiD-BCF.git"
ENGINE_SUBDIR = os.path.join("DiD-BCF", "Simulation_Studies_Revision")

def _locate_root():
    try:
        import did_bcf_revision  # noqa: F401
        return os.path.dirname(os.path.dirname(did_bcf_revision.__file__))
    except Exception:
        pass
    parent = os.path.abspath(os.path.join(os.getcwd(), ".."))
    if os.path.isdir(os.path.join(parent, "did_bcf_revision")):
        return parent
    if not os.path.isdir("DiD-BCF"):
        import subprocess
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL], check=True)
    return os.path.abspath(ENGINE_SUBDIR)

ROOT = _locate_root()
sys.path.insert(0, ROOT)
print("Using DiD-BCF engine at:", ROOT)

from did_bcf_revision.runner import run_named
from did_bcf_revision.metrics import (compute_metrics, plain_vs_corrected,
                                      surface_metrics)

Using DiD-BCF engine at: /content/DiD-BCF/Simulation_Studies_Revision


In [3]:
REPS = 100      # replications (lower for a quick smoke test)
JOBS = 1        # parallel reps (keep 1 on a single-core/GPU Colab)

bcf_params = dict(num_gfr=50, num_mcmc=500, keep_every=5, num_chains=3)

summaries = run_named(
    "B1_null",
    linearity_degree=2,
    reps=REPS,
    jobs=JOBS,
    bcf_params=bcf_params,
    prop_method="logit",   # pilot propensity for the posterior correction
    n_splits=2,            # cross-fitting folds for the correction
)
summaries.head()

[B1_null_lin_2] canonical DGP | N=(200,) | reps=100 | 100 fits | jobs=1


B1_null: 100%|██████████| 100/100 [6:07:57<00:00, 220.78s/fit]

[B1_null_lin_2] wrote 2000 rows -> /content/DiD-BCF/Simulation_Studies_Revision/Results/summaries_B1_null_lin_2.csv


,dgp,setting,linearity_degree,N,rep,estimand_type,estimand_id,g,t,k,...,p_bayes,surf_rmse,surf_mae,surf_n,surf_mape,surf_cover95,surf_len95,surf_cover90,surf_len90,true
0,canonical,B1_null,2,200,0,GATT,g=4_t=4,4.0,4.0,0.0,...,0.304,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0
1,canonical,B1_null,2,200,0,GATT,g=4_t=5,4.0,5.0,1.0,...,0.314,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0
2,canonical,B1_null,2,200,0,GATT,g=4_t=6,4.0,6.0,2.0,...,0.438,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0
3,canonical,B1_null,2,200,0,GATT,g=4_t=7,4.0,7.0,3.0,...,0.466,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0
4,canonical,B1_null,2,200,0,ES,k=0,NaN,NaN,0.0,...,0.304,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0


In [4]:
# Decomposed metrics: bias, MC SD/variance, RMSE, MAE, MAPE, coverage 90/95,
# interval length, calibration ratio (avg_post_sd/emp_sd), size/power and their
# Monte-Carlo SEs -- for plain AND corrected DiD-BCF.
metrics = compute_metrics(summaries)
plain_vs_corrected(metrics)

,dgp,setting,linearity_degree,N,estimand_type,estimand_id,role,bias_corrected,bias_plain,cover90_corrected,...,len95_corrected,len95_plain,mae_corrected,mae_plain,reject05_corrected,reject05_plain,rmse_corrected,rmse_plain,sd_ratio_corrected,sd_ratio_plain
0,canonical,B1_null,2,200,ATT,ATT,size,0.003824,0.015318,0.47,...,0.702686,1.142890,0.326047,0.039553,0.44,0.0,0.395238,0.052731,0.450114,5.357091
1,canonical,B1_null,2,200,ES,k=0,size,-0.015460,-0.000403,0.62,...,0.885989,1.012954,0.341623,0.006503,0.34,0.0,0.412836,0.008020,0.544065,28.337732
2,canonical,B1_null,2,200,ES,k=1,size,0.007812,0.013026,0.60,...,0.874620,1.188285,0.323255,0.045267,0.28,0.0,0.390416,0.059795,0.565860,4.827516
3,canonical,B1_null,2,200,ES,k=2,size,0.008424,0.022111,0.60,...,0.898978,1.310482,0.345572,0.059840,0.33,0.0,0.418177,0.079729,0.545163,4.132011
4,canonical,B1_null,2,200,ES,k=3,size,0.014519,0.026538,0.63,...,0.887449,1.421587,0.321958,0.066967,0.29,0.0,0.402276,0.091046,0.558903,3.989267
5,canonical,B1_null,2,200,GATT,g=4_t=4,size,-0.015460,-0.000403,0.62,...,0.885989,1.012954,0.341623,0.006503,0.34,0.0,0.412836,0.008020,0.544065,28.337732
6,canonical,B1_null,2,200,GATT,g=4_t=5,size,0.007812,0.013026,0.60,...,0.874620,1.188285,0.323255,0.045267,0.28,0.0,0.390416,0.059795,0.565860,4.827516
7,canonical,B1_null,2,200,GATT,g=4_t=6,size,0.008424,0.022111,0.60,...,0.898978,1.310482,0.345572,0.059840,0.33,0.0,0.418177,0.079729,0.545163,4.132011
8,canonical,B1_null,2,200,GATT,g=4_t=7,size,0.014519,0.026538,0.63,...,0.887449,1.421587,0.321958,0.066967,0.29,0.0,0.402276,0.091046,0.558903,3.989267


## CATT-surface metrics (the paper's headline RMSE/MAE/MAPE)

Within-replication RMSE/MAE/MAPE over the *individual* treated observations
(mean +/- SD across runs) plus the *pointwise* CATT coverage -- the evidence
that DiD-BCF recovers the heterogeneous effect that GATT-only methods cannot.

In [5]:
surface_metrics(summaries)

/content/DiD-BCF/Simulation_Studies_Revision/did_bcf_revision/metrics.py:190: RuntimeWarning: Mean of empty slice
  rec[f"{c}_mean"] = float(np.nanmean(vals)) if vals.size else np.nan


,dgp,setting,linearity_degree,N,method,n_reps,avg_n_treated_obs,surf_rmse_mean,surf_rmse_sd,surf_mae_mean,...,surf_mape_mean,surf_mape_sd,surf_cover90_mean,surf_cover90_sd,surf_cover95_mean,surf_cover95_sd,surf_len90_mean,surf_len90_sd,surf_len95_mean,surf_len95_sd
0,canonical,B1_null,2,200,corrected,100,401.24,0.345998,0.213614,0.333102,...,NaN,NaN,0.6125,0.408828,0.69,0.376252,0.73747,0.088106,0.886759,0.110979
1,canonical,B1_null,2,200,plain,100,401.24,0.055118,0.039388,0.044660,...,NaN,NaN,1.0000,0.000000,1.00,0.000000,0.98566,0.056203,1.236795,0.071611
